# 同じ行が何度も出てくると、何がいけない？

このノートは、有用性の観点 $U_\text{valid}$（妥当性）の中にある**重複行（no_dup）**を扱います（ルールブック §6.1.4）。

**ここで採点する観点**

- 全列の値がまったく同じ行が、何度も現れていないか。そうしたデータで人数や割合を集計すると、同じ人を何度も数えることになり、本物の $B$ で集計した値と違ってきます。

所要 5 分。

## 0. 準備

練習データ（`participant_data/B_practice.csv`）を使います。

In [1]:
import warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# 配布データの場所を探す（リポジトリの配置でも、zip を展開しただけの配置でも動くように）
DATA = next(p for p in (Path('..'), Path('../participant_data'), Path('participant_data'))
            if (p / 'B_practice.csv').exists())

from pwscup2026.kit import selfscore as ss        # CodaBench と同じ採点器

B_ref, REF, CFG, info = ss.load_local_reference(DATA)   # 採点器が使う参照値とパラメータ

# 採点と同じ読み方で読む（float_precision='round_trip'）。既定の読み方だと小数の末尾の桁がずれ、
# 値を変えていない列でも点数がごくわずかに下がることがあります。
B = pd.read_csv(DATA / 'B_practice.csv', float_precision='round_trip')

def score(C: pd.DataFrame) -> dict:
    # C を採点して {utility, facets, detail} を返す（CodaBench と同じ採点器）
    return ss.self_score(C.copy(), B_ref, REF, CFG)

print(f'B: {B.shape[0]} 行 × {B.shape[1]} 列')

B: 1049 行 × 14 列


## 1. 測り方

1. $C$ の全列（13列）の値が一致する行を、重複とみなします。
2. 重複している行の割合 $r=(1{,}049-\text{相異なる行数})/1{,}049$ を求めます。提出する $C$ の行数は、$B$ と同じ 1,049 行に決まっています（ルールブック §5.1）。
3. $\text{no\_dup}=1-\operatorname{clip}(r/0.2,\,0,\,1)$。重複の割合が 20% 以上で 0 点です。

ルールブックの式どおりに計算して、採点器の値と一致することを確かめます。本物の $B$ には、完全に同じ行はありません。

このノートの $C$ は $B$ を写して作るので、`record_id` 列が残っています。下の計算では `record_id` を除いて比べます（提出する `C.csv` にはこの列がありません）。

In [2]:
def dup_by_formula(C):
    # ルールブック §6.1.4 の重複行を式どおりに計算する
    cols = [c for c in C.columns if c != 'record_id']
    n = len(C)
    n_dup = n - len(C[cols].drop_duplicates())
    r = n_dup / n
    return {'重複している行数': n_dup, '重複の割合 r': r, 'no_dup': float(1 - np.clip(r / 0.2, 0, 1))}

print('式どおりに計算:', dup_by_formula(B))
print('採点器の値    :', score(B)['detail']['U_valid']['no_dup'])

式どおりに計算: {'重複している行数': 0, '重複の割合 r': 0.0, 'no_dup': 1.0}


採点器の値    : 1.0


## 2. 意味

- 1,049 行のデータは、1,049 人ぶんの別々の観測があるという前提で分析されます。同じ行が並ぶと、その人数ぶんの観測があるように見えますが、実際には同じ1人を何度も数えています。
- 重複は、行を複製したときだけでなく、値を粗く丸めたときにも生まれることがあります。
- $U_\text{valid}$ は4つの指標（打ち切り整合・関係の非捏造〔県・性別・喫煙による差を本物より強めていないか〕・重複行・行の新規性）の**平均**なので、no_dup だけで $U_\text{valid}$ が下がる幅は最大 0.25 です。

## 3. 壊してみる

### 3.1 行を複製する

$B$ の 5%・10%・20% の行を、ほかの行の複製に置き換えます（行数は変えません）。

In [3]:
def replace_with_copies(df, p, seed=0):
    rng = np.random.default_rng(seed)
    out = df.copy()
    n = len(out); k = int(round(p * n))
    cols = [c for c in out.columns if c != 'record_id']
    dst = rng.choice(n, k, replace=False)                            # 置き換える行
    src = rng.choice(np.setdiff1d(np.arange(n), dst), k, replace=True)  # 複製のもとにする行
    out.loc[out.index[dst], cols] = out.loc[out.index[src], cols].to_numpy()
    return out

rows = {}
for name, C in {'加工なし': B, '5% を複製に': replace_with_copies(B, 0.05),
                '10% を複製に': replace_with_copies(B, 0.10), '20% を複製に': replace_with_copies(B, 0.20)}.items():
    o = score(C)
    rows[name] = {**dup_by_formula(C), '採点器の no_dup': o['detail']['U_valid']['no_dup'],
                  'U_valid': o['facets']['U_valid'], 'U': o['utility']}
t = pd.DataFrame(rows).T.round(4)
t['重複している行数'] = t['重複している行数'].astype(int)
t

,重複している行数,重複の割合 r,no_dup,採点器の no_dup,U_valid,U
加工なし,0,0.0000,1.0000,1.0000,1.0000,1.0000
5% を複製に,52,0.0496,0.7521,0.7521,0.9376,0.8297
10% を複製に,105,0.1001,0.4995,0.4995,0.8737,0.8075
20% を複製に,210,0.2002,0.0000,0.0000,0.7498,0.7498


### 結果の解釈

- 置き換えた割合が、ほぼそのまま重複の割合 $r$ になります。no_dup は 5% で 0.75、10% で 0.50、20% で 0 です。
- $U_\text{valid}$ は4つの平均なので、20% でも約 0.75 です。
- 5% と 10% で $U$ が $U_\text{valid}$ より低いのは、特定タスクでの結果忠実度（$U_\text{spec}$）のほうが低いためです。複製された人の発症の有無と時点が何度も数えられるので、分析の結果も本物からずれます。

### 3.2 値を粗く丸める

連続値の列を大きな刻みに丸めます（age は 10 歳、BMI は 5、SBP は 20、TG は 50、HDL・ALT・FPG は 20 刻み。値域は `schema.json` の範囲に収めます）。行を複製していなくても、重複が生まれます。

In [4]:
import json
RANGES = json.loads((DATA / 'schema.json').read_text())['ranges']   # 値域（外に出ると提出が棄却されます）

def coarse_round(df):
    out = df.copy()
    for col, step in (('age', 10), ('BMI', 5), ('SBP', 20), ('TG', 50), ('HDL', 20), ('ALT', 20), ('FPG', 20)):
        out[col] = np.round(out[col] / step) * step
        if col in RANGES:
            out[col] = out[col].clip(*RANGES[col])
    return out

C_round = coarse_round(B)
o = score(C_round)
print(dup_by_formula(C_round))
print('採点器の no_dup:', round(o['detail']['U_valid']['no_dup'], 4), '  U_valid:', round(o['facets']['U_valid'], 4))

cols = [c for c in C_round.columns if c != 'record_id']
dups = C_round[C_round.duplicated(cols, keep=False)]
print('重複に関わった行の onset・death の内訳:', dups.groupby(['onset', 'death']).size().to_dict())
groups = dups.groupby(cols).size().sort_values(ascending=False)
print(f'\n同じ値の組が2行以上ある組: {len(groups)} 組。多い順に3組:')
groups.head(3).rename('行数').reset_index()

{'重複している行数': 24, '重複の割合 r': 0.022878932316491896, 'no_dup': 0.8856053384175405}
採点器の no_dup: 0.8856   U_valid: 0.9711
重複に関わった行の onset・death の内訳: {(0, 0): 44}

同じ値の組が2行以上ある組: 20 組。多い順に3組:


,age,sex,prefecture,BMI,SBP,TG,HDL,ALT,smoking,FPG,time,onset,death,行数
0,60.0,0,神奈川県,25.0,120.0,50.0,40.0,20.0,0,80.0,9.998631,0,0,4
1,50.0,0,神奈川県,20.0,100.0,50.0,40.0,0.1,0,80.0,9.998631,0,0,3
2,50.0,0,神奈川県,20.0,120.0,50.0,40.0,20.0,0,100.0,9.998631,0,0,3


### 結果の解釈

- 行を複製していなくても、24 行（2.3%）が重複になり、no_dup は 0.89 です。
- 重複したのは、どれも発症も死亡もしていない人です。この人たちは `time` が全員同じ値なので、ほかの列を丸めて値が揃うと、全列が一致しやすくなります。

## 4. 自分の C で確かめる

重複行は、採点結果の `detail` → `U_valid` → `no_dup` です。`detail` → `U_valid` → `no_dup_detail` に、重複の割合 `dup_rate` と行数 `n_dup_rows` が入っています。

In [5]:
score(C_round)['detail']['U_valid']['no_dup_detail']

{'dup_rate': 0.022878932316491896, 'n_dup_rows': 24, 'tol': 0.0, 'scale': 0.2}

自分の `C.csv` を、`starter/` と `participant_data/` が並んでいるフォルダに置き、そのフォルダで次のコマンドを実行すると、自己採点の `--json` の出力で同じ値を見られます。

```sh
cd starter
uv run python -m pwscup2026.kit.selfscore ../C.csv --dist ../participant_data --json
```

Docker を使う場合:

```sh
docker run --rm -v "$PWD":/w hajimeono/pwscup2026-kit:main-process-20260912 score /w/C.csv --dist /w/participant_data --json
```